In [1]:
lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"
job_id = 104

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 3, Finished, Available, Finished, False)

In [2]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_silver_db
""")

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 4, Finished, Available, Finished, False)

DataFrame[]

In [3]:
metadata_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_metadata_db.landing_metadata_tbl
WHERE JobID = 101
""")

metadata_list = metadata_df.collect()



StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 5, Finished, Available, Finished, False)

In [4]:
# spark.sql(f"""
# SHOW TABLES IN {lakehouse_name}.{env_name}_stage_db
# """).show(truncate=False)

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 6, Finished, Available, Finished, False)

In [5]:
stage_dfs = {}

for row in metadata_list:

    table_name = row["BronzeTableName"]

    print("=" * 60)
    print(f"Reading : {table_name}")

    df = spark.table(
        f"{lakehouse_name}.{env_name}_stage_db.{table_name}"
    )

    stage_dfs[table_name] = df

    # print("Rows :", df.count())

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 7, Finished, Available, Finished, False)

Reading : order_items
Reading : customers
Reading : products
Reading : orders


In [6]:
from pyspark.sql.functions import to_date, upper, trim

transformed_dfs = {}

for table_name, df in stage_dfs.items():

    print("=" * 60)
    print(f"Transforming : {table_name}")

    # Customers
# Customers
    if table_name == "customers":

        df = df.withColumn(
            "SignupDate",
            to_date("SignupDate", "yyyy-MM-dd")
        )

        df = df.withColumn(
            "IsPremiumCustomer",
            upper(trim("IsPremiumCustomer"))
        )

    # Orders
    elif table_name == "orders":

        df = df.withColumn(
            "OrderDate",
            to_date("OrderDate", "yyyy/MM/dd")
        )

        df = df.withColumn(
            "ShippedDate",
            to_date("ShippedDate", "yyyy/MM/dd")
        )

        df = df.withColumn(
            "DeliveredDate",
            to_date("DeliveredDate", "yyyy/MM/dd")
        )

    # Products
    elif table_name == "products":

        if "IsDiscounted" in df.columns:
            df = df.withColumn(
                "IsDiscounted",
                upper(trim("IsDiscounted"))
            )

    # Remove duplicate records
    df = df.dropDuplicates()

    transformed_dfs[table_name] = df

    # print("Rows :", df.count())

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 8, Finished, Available, Finished, False)

Transforming : order_items
Transforming : customers
Transforming : products
Transforming : orders


In [7]:
for table_name, df in transformed_dfs.items():

    df.limit(0).write \
        .mode("ignore") \
        .saveAsTable(
            f"{lakehouse_name}.{env_name}_silver_db.{table_name}"
        )

    # print(f"{table_name} Silver table ready")

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 9, Finished, Available, Finished, False)

In [8]:
primary_keys = {
    "customers": ["CustomerID"],
    "orders": ["OrderID"],
    "products": ["ProductID"],
    "order_items": ["OrderID", "ProductID"]
}

print(primary_keys)

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 10, Finished, Available, Finished, False)

{'customers': ['CustomerID'], 'orders': ['OrderID'], 'products': ['ProductID'], 'order_items': ['OrderID', 'ProductID']}


In [9]:
for table_name, df in transformed_dfs.items():

    df.createOrReplaceTempView(f"{table_name}_stage")

    print(f"{table_name}_stage view created")

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 11, Finished, Available, Finished, False)

order_items_stage view created
customers_stage view created
products_stage view created
orders_stage view created


In [10]:
for table_name, pk_columns in primary_keys.items():

    print("=" * 60)
    print(f"Merging : {table_name}")

    columns = transformed_dfs[table_name].columns

    merge_condition = " AND ".join(
        [f"target.{pk} = source.{pk}" for pk in pk_columns]
    )

    update_clause = ",\n".join(
        [
            f"target.{c} = source.{c}"
            for c in columns
            if c not in pk_columns
        ]
    )

    insert_columns = ", ".join(columns)

    insert_values = ", ".join(
        [f"source.{c}" for c in columns]
    )

    merge_sql = f"""
    MERGE INTO {lakehouse_name}.{env_name}_silver_db.{table_name} AS target
    USING {table_name}_stage AS source
    ON {merge_condition}

    WHEN MATCHED THEN
        UPDATE SET
        {update_clause}

    WHEN NOT MATCHED THEN
        INSERT ({insert_columns})
        VALUES ({insert_values})
    """

    spark.sql(merge_sql)

    # print(f"{table_name} merged successfully")

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 12, Finished, Available, Finished, False)

Merging : customers
Merging : orders
Merging : products
Merging : order_items


In [12]:
for table_name in transformed_dfs.keys():

    cnt = spark.sql(f"""
    SELECT COUNT(*) AS TotalRows
    FROM {lakehouse_name}.{env_name}_silver_db.{table_name}
    """)

    print("=" * 60)
    print(table_name)

    # cnt.show()

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 14, Finished, Available, Finished, False)

order_items
customers
products
orders


In [13]:
from datetime import datetime
from pyspark.sql.types import *

audit_rows = []

for table_name in transformed_dfs.keys():

    row_count = spark.table(
        f"{lakehouse_name}.{env_name}_silver_db.{table_name}"
    ).count()

    audit_rows.append(
        (
            104,
            "Stage_to_Silver",
            table_name,
            "Success",
            row_count,
            datetime.now(),
            datetime.now(),
            "Silver Load Completed"
        )
    )

schema = StructType([
    StructField("JobID", IntegerType(), True),
    StructField("JobName", StringType(), True),
    StructField("TableName", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("RowsProcessed", IntegerType(), True),
    StructField("StartTime", TimestampType(), True),
    StructField("EndTime", TimestampType(), True),
    StructField("Message", StringType(), True)
])

audit_df = spark.createDataFrame(audit_rows, schema)

audit_df.write \
.mode("append") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_audit_db.job_event_log_tbl"
)

print("Silver Audit Updated Successfully")

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 15, Finished, Available, Finished, False)

Silver Audit Updated Successfully


In [18]:
for table_name, pk_columns in primary_keys.items():

    print("=" * 60)
    print(f"Checking {table_name}")

    group_by_cols = ", ".join(pk_columns)

    spark.sql(f"""
        SELECT {group_by_cols}, COUNT(*) AS cnt
        FROM {table_name}_stage
        GROUP BY {group_by_cols}
        HAVING COUNT(*) > 1
    """)

StatementMeta(, bad2467d-99a4-4e85-abe8-a2587e290106, 22, Finished, Available, Finished, False)

Checking customers
Checking orders
Checking products
Checking order_items
